# Report

> Professional reporting for benchmark results

In [13]:
#| default_exp report

In [14]:
#| export
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Any
from pathlib import Path

from fasterbench.benchmark import BenchmarkResult
from fasterbench.core import _fmt_human, _section
from fasterbench.plot import create_radar_plot

## Metric Configuration

Data-driven configuration for each metric type, following the established pattern in `LayerProfiler`.

In [15]:
#| export
_METRIC_CONFIG: dict[str, dict[str, Any]] = {
    "params": {
        "label": "Parameters",
        "unit": "",
        "format": _fmt_human,
        "lower_is_better": True,
        "extract": lambda r: r.size.num_params if r.size else None,
    },
    "size_mib": {
        "label": "Model Size",
        "unit": "MiB",
        "format": lambda v: f"{v:.2f}",
        "lower_is_better": True,
        "extract": lambda r: r.size.size_mib if r.size else None,
    },
    "latency_cpu": {
        "label": "Latency (CPU)",
        "unit": "ms",
        "format": lambda v: f"{v:.2f}",
        "lower_is_better": True,
        "extract": lambda r: r.speed.get("cpu", None) and r.speed["cpu"].mean_ms,
    },
    "latency_cuda": {
        "label": "Latency (CUDA)",
        "unit": "ms",
        "format": lambda v: f"{v:.2f}",
        "lower_is_better": True,
        "extract": lambda r: r.speed.get("cuda", None) and r.speed["cuda"].mean_ms,
    },
    "throughput_cpu": {
        "label": "Throughput (CPU)",
        "unit": "inf/s",
        "format": lambda v: f"{v:.1f}",
        "lower_is_better": False,
        "extract": lambda r: r.speed.get("cpu", None) and r.speed["cpu"].throughput_s,
    },
    "throughput_cuda": {
        "label": "Throughput (CUDA)",
        "unit": "inf/s",
        "format": lambda v: f"{v:.1f}",
        "lower_is_better": False,
        "extract": lambda r: r.speed.get("cuda", None) and r.speed["cuda"].throughput_s,
    },
    "macs": {
        "label": "MACs",
        "unit": "M",
        "format": lambda v: f"{v:.1f}",
        "lower_is_better": True,
        "extract": lambda r: r.compute.macs_m if r.compute and r.compute.macs_available else None,
    },
    "memory_cpu": {
        "label": "Memory (CPU)",
        "unit": "MiB",
        "format": lambda v: f"{v:.2f}",
        "lower_is_better": True,
        "extract": lambda r: r.memory.get("cpu", None) and r.memory["cpu"].peak_mib,
    },
    "memory_cuda": {
        "label": "Memory (CUDA)",
        "unit": "MiB",
        "format": lambda v: f"{v:.2f}",
        "lower_is_better": True,
        "extract": lambda r: r.memory.get("cuda", None) and r.memory["cuda"].peak_mib,
    },
    "energy_cpu": {
        "label": "Energy (CPU)",
        "unit": "Wh",
        "format": lambda v: f"{v:.4f}",
        "lower_is_better": True,
        "extract": lambda r: r.energy.get("cpu", None) and r.energy["cpu"].energy_wh,
    },
}

## ReportMetricDelta

Dataclass for tracking metric changes between two benchmark results.

In [16]:
#| export
@dataclass(slots=True)
class ReportMetricDelta:
    """Represents the change in a metric between two benchmark results."""
    name: str           # metric key (e.g., "latency_cpu")
    label: str          # display label (e.g., "Latency (CPU)")
    before: float       # value before optimization
    after: float        # value after optimization
    delta: float        # absolute change (after - before)
    delta_pct: float    # percentage change
    improved: bool      # whether change is an improvement (direction-aware)
    unit: str           # unit for display
    
    def as_dict(self) -> dict[str, Any]:
        """Convert to dictionary for serialization."""
        return asdict(self)

## Helper Functions

In [17]:
#| export
def _generate_css() -> str:
    """Generate clean, professional CSS for HTML reports."""
    return """
<style>
    .report-container {
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, sans-serif;
        max-width: 900px;
        margin: 0 auto;
        padding: 40px;
        color: #333;
        line-height: 1.6;
    }
    .report-title {
        font-size: 28px;
        font-weight: 600;
        text-align: center;
        margin-bottom: 10px;
        color: #1a1a1a;
    }
    .report-subtitle {
        text-align: center;
        color: #666;
        margin-bottom: 30px;
    }
    .section-title {
        font-size: 18px;
        font-weight: 600;
        margin-top: 30px;
        margin-bottom: 15px;
        padding-bottom: 8px;
        border-bottom: 2px solid #e0e0e0;
        color: #1a1a1a;
    }
    .metrics-table {
        width: 100%;
        border-collapse: collapse;
        margin: 20px 0;
        font-size: 14px;
    }
    .metrics-table th {
        background: #f8f9fa;
        padding: 12px 16px;
        text-align: left;
        font-weight: 600;
        border-bottom: 2px solid #dee2e6;
    }
    .metrics-table td {
        padding: 12px 16px;
        border-bottom: 1px solid #e9ecef;
    }
    .metrics-table tr:hover {
        background: #f8f9fa;
    }
    .value-cell {
        font-family: 'SF Mono', Consolas, monospace;
        text-align: right;
    }
    .change-positive {
        color: #28a745;
        font-weight: 500;
    }
    .change-negative {
        color: #dc3545;
        font-weight: 500;
    }
    .improvement-icon {
        margin-left: 4px;
    }
    .chart-container {
        margin: 30px 0;
        text-align: center;
    }
    .model-info {
        background: #f8f9fa;
        padding: 15px 20px;
        border-radius: 8px;
        margin-bottom: 20px;
    }
    .model-info p {
        margin: 5px 0;
    }
    .description {
        color: #666;
        font-style: italic;
        margin-bottom: 20px;
    }
</style>
"""

In [18]:
#| export
def _improvement_indicator(improved: bool) -> str:
    """Return checkmark or X based on improvement status."""
    return "\u2713" if improved else "\u2717"  # ✓ or ✗

In [19]:
#| export
def _format_value_with_unit(value: float, cfg: dict) -> str:
    """Format a metric value with its unit."""
    formatted = cfg["format"](value)
    if cfg["unit"]:
        return f"{formatted} {cfg['unit']}"
    return formatted

In [20]:
#| export
def _extract_metrics(result: BenchmarkResult) -> dict[str, float | None]:
    """Extract all available metrics from a BenchmarkResult."""
    metrics = {}
    for key, cfg in _METRIC_CONFIG.items():
        try:
            value = cfg["extract"](result)
            metrics[key] = value
        except (AttributeError, TypeError, KeyError):
            metrics[key] = None
    return metrics

## Report Class

Single-model report for presenting benchmark results professionally.

In [21]:
#| export
class Report:
    """Professional report for a single model's benchmark results."""
    
    VALID_FORMATS = frozenset({"html", "markdown"})
    
    def __init__(
        self,
        result: BenchmarkResult,  # benchmark result to report on
        *,
        model_name: str = "Model",  # name for display
        description: str = "",      # optional description
    ):
        self.result = result
        self.model_name = model_name
        self.description = description
        self._metrics = _extract_metrics(result)
    
    def summary(self) -> None:
        """Print a formatted console summary."""
        width = 60
        print("=" * width)
        print(f"{self.model_name:^{width}}")
        print("=" * width)
        
        if self.description:
            print(f"\n{self.description}\n")
        
        print(_section("Metrics", width))
        
        for key, value in self._metrics.items():
            if value is not None:
                cfg = _METRIC_CONFIG[key]
                formatted = _format_value_with_unit(value, cfg)
                print(f"  {cfg['label']:.<30} {formatted}")
        print()
    
    def as_dict(self) -> dict[str, Any]:
        """Return report data as dictionary."""
        return {
            "model_name": self.model_name,
            "description": self.description,
            "metrics": {
                k: v for k, v in self._metrics.items() if v is not None
            },
            "raw_result": self.result.as_dict(),
        }
    
    def to_markdown(self, path: str | Path | None = None) -> str:
        """Generate markdown report.
        
        Args:
            path: If provided, write to file. Otherwise return string.
        
        Returns:
            Markdown string.
        """
        lines = [
            f"# {self.model_name} Benchmark Report",
            "",
        ]
        
        if self.description:
            lines.extend([f"*{self.description}*", ""])
        
        lines.extend([
            "## Metrics",
            "",
            "| Metric | Value |",
            "|--------|-------|",
        ])
        
        for key, value in self._metrics.items():
            if value is not None:
                cfg = _METRIC_CONFIG[key]
                formatted = _format_value_with_unit(value, cfg)
                lines.append(f"| {cfg['label']} | {formatted} |")
        
        content = "\n".join(lines)
        
        if path:
            Path(path).write_text(content)
        
        return content
    
    def to_html(
        self,
        path: str | Path | None = None,  # output file path (optional)
        *,
        include_charts: bool = True,     # include radar chart
    ) -> str:
        """Generate HTML report with optional charts.
        
        Args:
            path: If provided, write to file. Otherwise return string.
            include_charts: Whether to embed radar chart.
        
        Returns:
            HTML string.
        """
        # Build metrics table
        rows = []
        for key, value in self._metrics.items():
            if value is not None:
                cfg = _METRIC_CONFIG[key]
                formatted = _format_value_with_unit(value, cfg)
                rows.append(f"""
                <tr>
                    <td>{cfg['label']}</td>
                    <td class="value-cell">{formatted}</td>
                </tr>""")
        
        table_rows = "\n".join(rows)
        
        # Build chart if requested
        chart_html = ""
        if include_charts:
            try:
                fig = create_radar_plot([self.result], [self.model_name])
                chart_html = f"""
                <div class="chart-container">
                    {fig.to_html(include_plotlyjs='cdn', full_html=False)}
                </div>"""
            except Exception:
                chart_html = ""  # Skip chart on error
        
        description_html = f'<p class="description">{self.description}</p>' if self.description else ""
        
        html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{self.model_name} Benchmark Report</title>
    {_generate_css()}
</head>
<body>
    <div class="report-container">
        <h1 class="report-title">{self.model_name}</h1>
        <p class="report-subtitle">Benchmark Report</p>
        {description_html}
        
        <h2 class="section-title">Performance Metrics</h2>
        <table class="metrics-table">
            <thead>
                <tr>
                    <th>Metric</th>
                    <th style="text-align: right;">Value</th>
                </tr>
            </thead>
            <tbody>
                {table_rows}
            </tbody>
        </table>
        {chart_html}
    </div>
</body>
</html>"""
        
        if path:
            Path(path).write_text(html)
        
        return html

## ComparisonReport Class

Before/after comparison report for presenting optimization results.

In [22]:
#| export
class ComparisonReport:
    """Professional report comparing two benchmark results (before/after optimization)."""
    
    VALID_FORMATS = frozenset({"html", "markdown"})
    
    def __init__(
        self,
        before: BenchmarkResult,        # benchmark result before optimization
        after: BenchmarkResult,         # benchmark result after optimization
        *,
        before_name: str = "Original",  # display name for before model
        after_name: str = "Optimized",  # display name for after model
        title: str = "Model Compression Report",  # report title
    ):
        self.before = before
        self.after = after
        self.before_name = before_name
        self.after_name = after_name
        self.title = title
        
        self._before_metrics = _extract_metrics(before)
        self._after_metrics = _extract_metrics(after)
        self._deltas: list[ReportMetricDelta] | None = None
    
    @property
    def deltas(self) -> list[ReportMetricDelta]:
        """Compute and return metric deltas between before and after."""
        if self._deltas is not None:
            return self._deltas
        
        deltas = []
        for key, cfg in _METRIC_CONFIG.items():
            before_val = self._before_metrics.get(key)
            after_val = self._after_metrics.get(key)
            
            # Skip if either value is missing
            if before_val is None or after_val is None:
                continue
            
            delta = after_val - before_val
            delta_pct = (delta / before_val * 100) if before_val != 0 else 0.0
            
            # Determine if this is an improvement based on direction
            if cfg["lower_is_better"]:
                improved = delta < 0
            else:
                improved = delta > 0
            
            deltas.append(ReportMetricDelta(
                name=key,
                label=cfg["label"],
                before=before_val,
                after=after_val,
                delta=delta,
                delta_pct=delta_pct,
                improved=improved,
                unit=cfg["unit"],
            ))
        
        self._deltas = deltas
        return deltas
    
    def top_improvements(self, n: int = 5) -> list[ReportMetricDelta]:
        """Return top N improvements sorted by absolute percentage change."""
        improved = [d for d in self.deltas if d.improved]
        return sorted(improved, key=lambda d: abs(d.delta_pct), reverse=True)[:n]
    
    def summary(self) -> None:
        """Print a formatted console summary."""
        width = 65
        
        print("=" * width)
        print(f"{self.title:^{width}}")
        print("=" * width)
        print()
        print(f"Before: {self.before_name}")
        print(f"After:  {self.after_name}")
        print()
        
        # Executive summary table
        print(_section("Executive Summary", width))
        print(f"{'':20} {'Before':>15} {'After':>15} {'Change':>12}")
        
        for d in self.deltas:
            cfg = _METRIC_CONFIG[d.name]
            before_str = _format_value_with_unit(d.before, cfg)
            after_str = _format_value_with_unit(d.after, cfg)
            
            sign = "+" if d.delta_pct > 0 else ""
            indicator = _improvement_indicator(d.improved)
            change_str = f"{sign}{d.delta_pct:.1f}% {indicator}"
            
            print(f"  {d.label:.<18} {before_str:>15} {after_str:>15} {change_str:>12}")
        
        print()
        
        # Top improvements
        top = self.top_improvements()
        if top:
            print(_section("Top Improvements", width))
            for i, d in enumerate(top, 1):
                print(f"  {i}. {d.label}: {d.delta_pct:.1f}%")
        print()
    
    def as_dict(self) -> dict[str, Any]:
        """Return report data as dictionary."""
        return {
            "title": self.title,
            "before_name": self.before_name,
            "after_name": self.after_name,
            "deltas": [d.as_dict() for d in self.deltas],
            "before_metrics": self._before_metrics,
            "after_metrics": self._after_metrics,
        }
    
    def to_markdown(self, path: str | Path | None = None) -> str:
        """Generate markdown comparison report.
        
        Args:
            path: If provided, write to file. Otherwise return string.
        
        Returns:
            Markdown string.
        """
        lines = [
            f"# {self.title}",
            "",
            f"**Before:** {self.before_name}",
            f"**After:** {self.after_name}",
            "",
            "## Summary",
            "",
            "| Metric | Before | After | Change |",
            "|--------|--------|-------|--------|",
        ]
        
        for d in self.deltas:
            cfg = _METRIC_CONFIG[d.name]
            before_str = _format_value_with_unit(d.before, cfg)
            after_str = _format_value_with_unit(d.after, cfg)
            sign = "+" if d.delta_pct > 0 else ""
            indicator = _improvement_indicator(d.improved)
            change_str = f"{sign}{d.delta_pct:.1f}% {indicator}"
            lines.append(f"| {d.label} | {before_str} | {after_str} | {change_str} |")
        
        # Top improvements section
        top = self.top_improvements()
        if top:
            lines.extend([
                "",
                "## Top Improvements",
                "",
            ])
            for i, d in enumerate(top, 1):
                lines.append(f"{i}. **{d.label}**: {d.delta_pct:.1f}%")
        
        content = "\n".join(lines)
        
        if path:
            Path(path).write_text(content)
        
        return content
    
    def to_html(
        self,
        path: str | Path | None = None,  # output file path (optional)
        *,
        include_charts: bool = True,     # include radar chart comparison
    ) -> str:
        """Generate HTML comparison report with optional charts.
        
        Args:
            path: If provided, write to file. Otherwise return string.
            include_charts: Whether to embed radar chart comparison.
        
        Returns:
            HTML string.
        """
        # Build comparison table rows
        rows = []
        for d in self.deltas:
            cfg = _METRIC_CONFIG[d.name]
            before_str = _format_value_with_unit(d.before, cfg)
            after_str = _format_value_with_unit(d.after, cfg)
            
            sign = "+" if d.delta_pct > 0 else ""
            change_class = "change-positive" if d.improved else "change-negative"
            indicator = _improvement_indicator(d.improved)
            
            rows.append(f"""
                <tr>
                    <td>{d.label}</td>
                    <td class="value-cell">{before_str}</td>
                    <td class="value-cell">{after_str}</td>
                    <td class="value-cell {change_class}">{sign}{d.delta_pct:.1f}%<span class="improvement-icon">{indicator}</span></td>
                </tr>""")
        
        table_rows = "\n".join(rows)
        
        # Build top improvements list
        top = self.top_improvements()
        top_items = "".join([
            f"<li><strong>{d.label}:</strong> {d.delta_pct:.1f}%</li>"
            for d in top
        ])
        top_html = f"<ol>{top_items}</ol>" if top else ""
        
        # Build chart if requested
        chart_html = ""
        if include_charts:
            try:
                fig = create_radar_plot(
                    [self.before, self.after],
                    [self.before_name, self.after_name]
                )
                chart_html = f"""
                <div class="chart-container">
                    {fig.to_html(include_plotlyjs='cdn', full_html=False)}
                </div>"""
            except Exception:
                chart_html = ""  # Skip chart on error
        
        html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{self.title}</title>
    {_generate_css()}
</head>
<body>
    <div class="report-container">
        <h1 class="report-title">{self.title}</h1>
        
        <div class="model-info">
            <p><strong>Before:</strong> {self.before_name}</p>
            <p><strong>After:</strong> {self.after_name}</p>
        </div>
        
        <h2 class="section-title">Executive Summary</h2>
        <table class="metrics-table">
            <thead>
                <tr>
                    <th>Metric</th>
                    <th style="text-align: right;">Before</th>
                    <th style="text-align: right;">After</th>
                    <th style="text-align: right;">Change</th>
                </tr>
            </thead>
            <tbody>
                {table_rows}
            </tbody>
        </table>
        
        <h2 class="section-title">Top Improvements</h2>
        {top_html}
        
        {chart_html}
    </div>
</body>
</html>"""
        
        if path:
            Path(path).write_text(html)
        
        return html

## Example Usage

In [23]:
#| eval: false
import torch
import torch.nn as nn
from fasterbench import benchmark

# Create a simple model for testing
class SimpleModel(nn.Module):
    def __init__(self, hidden=100):
        super().__init__()
        self.fc1 = nn.Linear(100, hidden)
        self.fc2 = nn.Linear(hidden, 10)
    
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

# Benchmark original model
model = SimpleModel(hidden=100)
sample = torch.randn(1, 100)
result = benchmark(model, sample, metrics=['size', 'speed', 'compute'])

# Single model report
report = Report(result, model_name='SimpleModel')
report.summary()

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
                        SimpleModel                         
═══ Metrics ═════════════════════════════════════════════════════
  Parameters.................... 11.11K
  Model Size.................... 0.04 MiB
  Latency (CPU)................. 0.01 ms
  Latency (CUDA)................ 0.02 ms
  Throughput (CPU).............. 149533.5 inf/s
  Throughput (CUDA)............. 53089.4 inf/s
  MACs.......................... 0.0 M



In [24]:
#| eval: false
# Comparison report
model_small = SimpleModel(hidden=50)  # "Optimized" smaller model
result_small = benchmark(model_small, sample, metrics=['size', 'speed', 'compute'])

comparison = ComparisonReport(
    result, result_small,
    before_name='Original (100 hidden)',
    after_name='Optimized (50 hidden)',
)
comparison.summary()

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
                    Model Compression Report                     

Before: Original (100 hidden)
After:  Optimized (50 hidden)

═══ Executive Summary ════════════════════════════════════════════════
                              Before           After       Change
  Parameters........          11.11K           5.56K     -50.0% ✓
  Model Size........        0.04 MiB        0.02 MiB     -47.4% ✓
  Latency (CPU).....         0.01 ms         0.01 ms      -4.6% ✓
  Latency (CUDA)....         0.02 ms         0.02 ms      -0.6% ✓
  Throughput (CPU)..  149533.5 inf/s  156799.2 inf/s      +4.9% ✓
  Throughput (CUDA).   53089.4 inf/s   53431.6 inf/s      +0.6% ✓
  MACs..............           0.0 M           0.0 M     -54.5% ✓

═══ Top Improvements ═════════════════════════════════════════════════
  1. MACs: -54.5%
  2. Parameters: -50.0%
  3. Model Size: -47.4%
  4. Throughput (CPU): 4.9%
  5. Latency (CPU): -4.6%



In [26]:
from fasterbench.report import Report, ComparisonReport, ReportMetricDelta

# Single model report
report = Report(result, model_name="ResNet-18")
report.summary()                    # Console output
report.to_html("report.html")       # HTML with embedded charts
report.to_markdown("report.md")     # Markdown tables

                         ResNet-18                          
═══ Metrics ═════════════════════════════════════════════════════
  Parameters.................... 11.11K
  Model Size.................... 0.04 MiB
  Latency (CPU)................. 0.01 ms
  Latency (CUDA)................ 0.02 ms
  Throughput (CPU).............. 149533.5 inf/s
  Throughput (CUDA)............. 53089.4 inf/s
  MACs.......................... 0.0 M



'# ResNet-18 Benchmark Report\n\n## Metrics\n\n| Metric | Value |\n|--------|-------|\n| Parameters | 11.11K |\n| Model Size | 0.04 MiB |\n| Latency (CPU) | 0.01 ms |\n| Latency (CUDA) | 0.02 ms |\n| Throughput (CPU) | 149533.5 inf/s |\n| Throughput (CUDA) | 53089.4 inf/s |\n| MACs | 0.0 M |'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()